<a href="https://colab.research.google.com/github/Raymondycp/AI_VirtualTryOn_Retail_Fashion/blob/main/Scraping_data_from_Uniqlo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install requests beautifulsoup4 selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.2/499.2 kB 22.9 MB/s eta 0:00:00


# [A]Extract All Product Links and Store Them in a DataFrame

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import pandas as pd
import time

In [ ]:
# Function to set up Selenium WebDriver
def setup_driver():
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')  # Run in headless mode (no browser window)
    options.add_argument('--disable-gpu')
    options.add_argument('--no-sandbox')
    return webdriver.Chrome(options=options)

# Function to scroll and extract all product links
def get_all_product_links(driver, base_url):
    driver.get(base_url)
    time.sleep(5)  # Wait for the initial page to load

    # Scroll to the bottom of the page until no more products are loaded
    last_height = driver.execute_script("return document.body.scrollHeight")
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3)  # Wait for new content to load

        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break  # Stop scrolling if no new content is loaded
        last_height = new_height

    # Parse the page source with BeautifulSoup
    soup = BeautifulSoup(driver.page_source, 'html.parser')

    # Extract all product links
    product_urls = set()
    for link in soup.find_all('a', class_='h-a-label'):
        href = link.get('href')
        if href and "product-detail" in href:
            full_url = f"https://www.uniqlo.com.hk{href}"
            product_urls.add(full_url)

    # Convert the set of URLs to a DataFrame
    df = pd.DataFrame(product_urls, columns=["Product URL"])
    return df

# Base URL for the product category
base_url = "https://www.uniqlo.com.hk/zh_HK/c/all-men-tops.html"

# Set up the WebDriver and get all product links
driver = setup_driver()
product_links_df = get_all_product_links(driver, base_url)
print(f"Found {len(product_links_df)} product links.")

Found 335 product links.


# Extract Images and Product Information for Each Link

In [ ]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time

def scroll_to_bottom(driver):
    """
    Scrolls to the bottom of the page to load all content.
    """
    last_height = driver.execute_script("return document.body.scrollHeight")
    while True:
        # Scroll to the bottom of the page
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3)  # Wait for new content to load

        # Calculate new scroll height and compare with last scroll height
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break  # Stop scrolling if no new content is loaded
        last_height = new_height

def get_product_details(driver, product_url):
    """
    Scrapes detailed information for a single product URL.
    """
    driver.get(product_url)
    time.sleep(5)  # Initial wait for the page to load

    # Scroll to the bottom of the page to load all content
    scroll_to_bottom(driver)

    # Parse the page source with BeautifulSoup
    soup = BeautifulSoup(driver.page_source, 'html.parser')

    # Extract product name
    try:
        product_name = driver.find_element(By.XPATH, '/html/body/div[2]/div/div[1]/div[2]/div/div/div/div/div[1]/div[4]/div[2]/div/div[1]/div[1]').text.strip()
    except Exception:
        product_name = "Name not found"

    # Extract all colors
    try:
        color_elements = soup.select('ul.h-clearfix.sku-select-colors li img')
        colors = [img.get('alt') for img in color_elements if img.get('alt')]
    except Exception:
        colors = ["Colors not found"]

    # Extract all sizes
    try:
        size_elements = soup.select('ul.h-clearfix.sku-select-sizes li span')
        sizes = [size.text.strip() for size in size_elements]
    except Exception:
        sizes = ["Sizes not found"]

    # Extract all product images
    try:
        product_images = []
        image_containers = soup.select('div.h-col.picture-viewer-item')
        for container in image_containers:
            img_tag = container.find('img', class_='picture-img')
            if img_tag and img_tag.get('src') and img_tag['src'].lower().endswith('.jpg'):
                product_images.append(img_tag['src'])
    except Exception:
        product_images = ["Product images not found"]

    # Extract all model images
    try:
        model_image_elements = soup.select('div.h-styling-item div.styling-image img.picture-img')
        model_images = [img['src'] for img in model_image_elements if img.get('src') and img['src'].lower().endswith('.jpg')]
    except Exception:
        model_images = ["Model images not found"]

    # Return the extracted details as a dictionary
    return {
        "Product URL": product_url,
        "Name": product_name,
        "Colors": colors,
        "Sizes": sizes,
        "Product Images": product_images,
        "Model Images": model_images
    }

# Create an empty list to store product details
product_details_list = []

# Define the maximum number of links to process (set to None to process all links)
max_links = 10  # Change this value to None or another number as needed

# Iterate over the product links and extract details
for index, row in product_links_df.iterrows():
    if max_links is not None and index >= max_links:
        break  # Stop processing after reaching the specified limit

    product_url = row["Product URL"]
    print(f"Scraping details for: {product_url}")
    details = get_product_details(driver, product_url)
    product_details_list.append(details)

# Convert the list of dictionaries to a DataFrame
product_details_df = pd.DataFrame(product_details_list)
print(f"Finished scraping details for {len(product_details_df)} products.")

Scraping details for: https://www.uniqlo.com.hk/zh_HK/product-detail.html?productCode=u0000000052081
Scraping details for: https://www.uniqlo.com.hk/zh_HK/product-detail.html?productCode=u0000000052614
Scraping details for: https://www.uniqlo.com.hk/zh_HK/product-detail.html?productCode=u0000000052596
Scraping details for: https://www.uniqlo.com.hk/zh_HK/product-detail.html?productCode=u0000000051435
Scraping details for: https://www.uniqlo.com.hk/zh_HK/product-detail.html?productCode=u0000000052456
Scraping details for: https://www.uniqlo.com.hk/zh_HK/product-detail.html?productCode=u0000000052910
Scraping details for: https://www.uniqlo.com.hk/zh_HK/product-detail.html?productCode=u0000000050695
Scraping details for: https://www.uniqlo.com.hk/zh_HK/product-detail.html?productCode=u0000000052722
Scraping details for: https://www.uniqlo.com.hk/zh_HK/product-detail.html?productCode=u0000000051767
Scraping details for: https://www.uniqlo.com.hk/zh_HK/product-detail.html?productCode=u0000

# Output the Results to a CSV File

In [ ]:
# Save the product details DataFrame to a CSV file
output_csv = "uniqlo_product_details.csv"
product_details_df.to_csv(output_csv, index=False, encoding='utf-8')

print(f"Product details saved to {output_csv}.")

Product details saved to uniqlo_product_details.csv.


In [ ]:
product_details_df['Product Images'][5]

['https://www.uniqlo.com.hk/hmall/test/u0000000052910/main/first/561/1.jpg',
 'https://www.uniqlo.com.hk/hmall/test/u0000000052910/main/other1/480/2.jpg']

## Preview at Html

In [ ]:
from IPython.display import HTML, display

def preview_product_details(product_details_df):
    """
    Generates a simplified HTML preview of the scraped product details and displays it in the notebook.
    """
    # Start generating HTML content
    html_content = """
    <style>
        body {
            font-family: Arial, sans-serif;
            margin: 20px;
        }
        .product {
            margin-bottom: 20px;
            border-bottom: 1px solid #ccc;
            padding-bottom: 20px;
        }
        .product h2 {
            margin: 0 0 10px 0;
        }
        .product img {
            width: 100px;
            height: 100px;
            margin-right: 10px;
            margin-bottom: 10px;
            border: 1px solid #ddd;
            border-radius: 4px;
            padding: 5px;
        }
        .product p {
            margin: 5px 0;
        }
    </style>
    <h1>Product Details Preview</h1>
    """

    # Iterate through each row in the DataFrame
    for index, row in product_details_df.iterrows():
        html_content += f"""
        <div class="product">
            <h2>{row['Name']}</h2>
            <p><strong>Product URL:</strong> <a href="{row['Product URL']}" target="_blank">{row['Product URL']}</a></p>
            <p><strong>Colors:</strong> {', '.join(row['Colors']) if isinstance(row['Colors'], list) else row['Colors']}</p>
            <p><strong>Sizes:</strong> {', '.join(row['Sizes']) if isinstance(row['Sizes'], list) else row['Sizes']}</p>

            <p><strong>Product Images:</strong></p>
            <div>
        """

        # Add product images
        for img_url in row['Product Images']:
            html_content += f'<img src="{img_url}" alt="Product Image">'

        html_content += """
            </div>

            <p><strong>Model Images:</strong></p>
            <div>
        """

        # Add model images
        for img_url in row['Model Images']:
            html_content += f'<img src="{img_url}" alt="Model Image">'

        html_content += """
            </div>
        </div>
        """

    # Display the generated HTML content
    display(HTML(html_content))

# Example usage
if __name__ == "__main__":
    # Assuming `product_details_df` contains the scraped data
    preview_product_details(product_details_df)

# Extracting info (Img to text)

In [ ]:
import pandas as pd
from transformers import pipeline
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import webbrowser

class ProductSearcher:
    """產品檢索器，用於根據自然語言查詢檢索相關產品"""

    def __init__(self, product_details):
        """
        初始化檢索器
        :param product_details: 產品詳細資訊，格式為 {url: {名稱: ..., 所有顏色: ..., 文字描述: ..., 所有官方圖片: ..., 所有穿搭圖片: ...}}
        """
        self.product_details = product_details
        self.nlp_model = pipeline("feature-extraction", model="distilbert-base-uncased")

    def _get_text_vectb or(self, text):
        """將文本轉換為向量"""
        return np.mean(self.nlp_model(text)[0], axis=0)

    def search(self, query, top_k=5):
        """
        根據自然語言查詢檢索相關產品
        :param query: 查詢內容（字符串）
        :param top_k: 返回最相關的結果數量
        :return: 檢索結果列表，格式為 [{"名稱": ..., "所有顏色": ..., "文字描述": ..., "所有官方圖片": ..., "所有穿搭圖片": ..., "相似度": ...}, ...]
        """
        # 將查詢轉換為向量
        query_vector = self._get_text_vector(query)

        results = []
        for url, details in self.product_details.items():
            # 將產品描述轉換為向量
            description_vector = self._get_text_vector(details["文字描述"])

            # 計算餘弦相似度
            similarity = cosine_similarity([query_vector], [description_vector])[0][0]

            # 將結果保存
            results.append({
                "名稱": details["名稱"],
                "所有顏色": eval(details["所有顏色"]),  # 將字符串轉換為列表
                "文字描述": details["文字描述"],
                "所有官方圖片": eval(details["所有官方圖片"]),  # 將字符串轉換為列表
                "所有穿搭圖片": eval(details["所有穿搭圖片"]),  # 將字符串轉換為列表
                "相似度": similarity
            })

        # 按相似度排序
        results.sort(key=lambda x: x["相似度"], reverse=True)
        return results[:top_k]  # 返回前 top_k 個最相關的結果

# 從 CSV 文件中讀取資料
def load_product_details_from_csv(csv_filename):
    """從 CSV 文件中讀取貨品詳細資訊"""
    df = pd.read_csv(csv_filename, index_col=0)
    product_details = df.to_dict(orient='index')
    return product_details

# 生成 HTML 頁面
def generate_html(results, query):
    """
    將檢索結果生成為 HTML 頁面
    :param results: 檢索結果列表
    :param query: 查詢內容
    :return: HTML 內容（字符串）
    """
    html_content = f"""
    <html>
    <head>
        <meta charset="UTF-8">
        <title>檢索結果: {query}</title>
        <style>
            body {{ font-family: Arial, sans-serif; margin: 20px; }}
            .result {{ margin-bottom: 20px; border: 1px solid #ddd; padding: 15px; border-radius: 5px; }}
            .result h3 {{ margin-top: 0; }}
            .images {{ display: flex; flex-wrap: wrap; gap: 10px; }}
            .images img {{ max-width: 200px; max-height: 200px; border-radius: 5px; }}
        </style>
    </head>
    <body>
        <h1>檢索結果: {query}</h1>
    """

    for i, result in enumerate(results):
        html_content += f"""
        <div class="result">
            <h3>{i+1}. {result['名稱']} (相似度: {result['相似度']:.4f})</h3>
            <p><strong>描述:</strong> {result['文字描述']}</p>
            <p><strong>顏色:</strong> {', '.join(result['所有顏色'])}</p>
            <div class="images">
        """

        # 添加官方圖片
        if result["所有官方圖片"]:
            html_content += "<h4>官方圖片:</h4>"
            for img_url in result["所有官方圖片"]:
                html_content += f'<img src="{img_url}" alt="官方圖片">'

        # 添加穿搭圖片
        if result["所有穿搭圖片"]:
            html_content += "<h4>穿搭圖片:</h4>"
            for img_url in result["所有穿搭圖片"]:
                html_content += f'<img src="{img_url}" alt="穿搭圖片">'

        html_content += """
            </div>
        </div>
        """

    html_content += """
    </body>
    </html>
    """
    return html_content

# 保存 HTML 文件並在瀏覽器中打開
def save_and_open_html(html_content, filename="results.html"):
    """將 HTML 內容保存為文件並在瀏覽器中打開"""
    with open(filename, "w", encoding="utf-8") as file:
        file.write(html_content)
    print(f"HTML 文件已保存為 {filename}")
    webbrowser.open(filename)

# 測試檢索功能並生成 HTML
def test_search_and_generate_html(product_details):
    """測試檢索功能並生成 HTML 頁面"""
    # 初始化檢索器
    searcher = ProductSearcher(product_details)

    # 輸入查詢
    query = input("請輸入查詢內容（例如：舒適的男士T恤）：")

    # 檢索相關產品
    results = searcher.search(query)

    # 生成 HTML 頁面
    html_content = generate_html(results, query)

    # 保存並打開 HTML 文件
    save_and_open_html(html_content)

# 主函數
def main():
    # 從 CSV 文件中讀取貨品詳細資訊
    csv_filename = "product_details.csv"
    product_details = load_product_details_from_csv(csv_filename)

    # 測試檢索功能並生成 HTML
    test_search_and_generate_html(product_details)

if __name__ == "__main__":
    main()

Device set to use cuda:0


請輸入查詢內容（例如：舒適的男士T恤）：黑色女裝
HTML 文件已保存為 results.html


In [ ]:
product_details = load_product_details_from_csv("product_details.csv")
searcher = ProductSearcher(product_details)
results = searcher.search("")
for result in results:
    print(result)

Device set to use cuda:0


{'名稱': '男女通用 PUFFTECH 壓線外套 472293', '所有顏色': ['黑色', '淺啡色', '綠色'], '文字描述': '商品說明\xa0\n※ 此商品特別尺碼為 XS，3XL※ 特別尺碼的實際庫存請依頁面顯之庫存為準使用最新纖維科技、輕盈保暖的高功能中棉「PUFFTECH」。· 約為頭髮 1/5 細的中空纖維，鎖住溫暖空氣，保暖效果佳。跣水耐用，且可手洗，可輕鬆保養。· 不過於休閒的菱形絎縫設計，適合多種場合穿著。· 適度的寬鬆版型，男女皆適穿。物料組成表側：100% 尼龍/填充物：100% 聚酯纖維/裡料：100% 尼龍/羅紋部分：85% 腈綸，14% 聚酯纖維，1% 氨綸/口袋襯裡：100% 尼龍洗滌方式手洗，不可乾洗※ 此商品退換貨詳情請參閱換貨、退貨及退款細則。※ 圖中展示之商品顏色以頁面實際庫存為準。', '所有官方圖片': ['https://www.uniqlo.com.hk/hmall/test/u0000000051539/main/first/561/1.jpg', 'https://www.uniqlo.com.hk/hmall/test/u0000000051539/main/other1/480/2.jpg', 'https://www.uniqlo.com.hk/hmall/test/u0000000051539/main/other2/480/3.jpg', 'https://www.uniqlo.com.hk/hmall/test/u0000000051539/main/other3/480/4.jpg', 'https://www.uniqlo.com.hk/hmall/test/u0000000051539/main/other4/480/5.jpg', 'https://www.uniqlo.com.hk/hmall/test/u0000000051539/main/other5/480/6.jpg', 'https://www.uniqlo.com.hk/hmall/test/u0000000051539/main/other6/480/7.jpg', 'https://www.uniqlo.com.hk/hmall/test/u0000000051539/main/other7/480/8.jpg

# [B]Extract 100 model photos and make description

In [ ]:
from selenium import webdriver
from bs4 import BeautifulSoup
import time

# 設置 Selenium 選項
options = webdriver.ChromeOptions()
options.add_argument('--headless')  # 無頭模式，不打開瀏覽器窗口
options.add_argument('--disable-gpu')
options.add_argument('--no-sandbox')

# 啟動瀏覽器
driver = webdriver.Chrome(options=options)

# 目標網址
url = "https://www.uniqlo.com.hk/hk/zh_HK/stylingbook/stylehint/men?colorId=BLACK"

# 打開網頁
driver.get(url)

# 等待頁面加載
time.sleep(5)  # 根據網絡情況調整等待時間

# 模擬滾動以觸發懶加載
for _ in range(10):  # 滾動次數根據頁面內容調整
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)  # 等待加載

# 獲取頁面源碼
page_source = driver.page_source

# 關閉 WebDriver
driver.quit()

# 使用 BeautifulSoup 解析頁面
soup = BeautifulSoup(page_source, 'html.parser')

# 查找所有目標圖片
image_divs = soup.find_all('div', class_='fr-ec-image')

# 提取圖片網址
image_urls = []
for div in image_divs:
    img_tag = div.find('img', class_='fr-ec-image__img')
    if img_tag and 'src' in img_tag.attrs:
        image_urls.append(img_tag['src'])

import pandas as pd

# Creating a DataFrame and writing to CSV
imagedf = pd.DataFrame(image_urls, columns=['image_urls'])

# 生成 HTML 文件
html_content = """
<!DOCTYPE html>
<html lang="zh-HK">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>UNIQLO 圖片預覽</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            background-color: #f4f4f4;
            padding: 20px;
        }
        .image-container {
            display: flex;
            flex-wrap: wrap;
            gap: 10px;
        }
        .image-container img {
            max-width: 200px;
            height: auto;
            border: 1px solid #ddd;
            border-radius: 4px;
            padding: 5px;
            background-color: #fff;
        }
    </style>
</head>
<body>
    <h1>UNIQLO 圖片預覽</h1>
    <div class="image-container">
"""

# 將圖片網址嵌入到 HTML 中
for img_url in image_urls[:100]:  # 只取前 100 張圖片
    html_content += f'<img src="{img_url}" alt="UNIQLO 圖片">\n'

# 結束 HTML 內容
html_content += """
    </div>
</body>
</html>
"""

# 將 HTML 內容寫入文件
with open("uniqlo_images_preview.html", "w", encoding="utf-8") as file:
    file.write(html_content)

print("HTML 文件已生成：uniqlo_images_preview.html")
print(imagedf.head())

HTML 文件已生成：uniqlo_images_preview.html
                                          image_urls
0  https://api.fastretailing.com/ugc/v1/uq/jp/SR_...
1  https://api.fastretailing.com/ugc/v1/uq/jp/SR_...
2  https://api.fastretailing.com/ugc/v1/uq/jp/SR_...
3  https://api.fastretailing.com/ugc/v1/uq/jp/SR_...
4  https://api.fastretailing.com/ugc/v1/uq/jp/SR_...


# Download images form imagedf


In [ ]:
import os
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed


# 创建一个目录来保存下载的图片
if not os.path.exists('images'):
    os.makedirs('images')

# 設定下載的數量
download_limit = 100  # 例如只下載前10個圖片

def download_image(index, url):
    try:
        # 发送 HTTP GET 请求获取图片
        response = requests.get(url, stream=True)
        response.raise_for_status()  # 检查请求是否成功

        # 保存图片到本地
        with open(f'images/image_{index}.jpg', 'wb') as file:
            for chunk in response.iter_content(chunk_size=8192):
                file.write(chunk)

        print(f"Downloaded image {index} from {url}")
        return True

    except requests.exceptions.RequestException as e:
        print(f"Failed to download image {index} from {url}: {e}")
        return False

# 使用多線程並行下載圖片
with ThreadPoolExecutor(max_workers=5) as executor:  # 你可以根據需要調整線程數量
    futures = []
    for index, url in enumerate(imagedf['image_urls'][:download_limit]):  # 只下載前 download_limit 個圖片
        futures.append(executor.submit(download_image, index, url))

    # 等待所有下載任務完成
    for future in as_completed(futures):
        future.result()

print("All images downloaded.")

Downloaded image 4 from https://api.fastretailing.com/ugc/v1/uq/jp/SR_IMAGES/ugc_stylehint_uq_jp_photo_250220_1563548_r-600-800
Downloaded image 1 from https://api.fastretailing.com/ugc/v1/uq/jp/SR_IMAGES/ugc_stylehint_uq_jp_photo_250220_1563452_r-600-800
Downloaded image 0 from https://api.fastretailing.com/ugc/v1/uq/jp/SR_IMAGES/ugc_stylehint_uq_jp_photo_250220_1563491_r-600-800
Downloaded image 3 from https://api.fastretailing.com/ugc/v1/uq/jp/SR_IMAGES/ugc_stylehint_uq_jp_photo_250220_1557673_r-600-800
Downloaded image 6 from https://api.fastretailing.com/ugc/v1/uq/jp/SR_IMAGES/ugc_stylehint_uq_jp_photo_250220_1563546_r-600-800
Downloaded image 5 from https://api.fastretailing.com/ugc/v1/uq/jp/SR_IMAGES/ugc_stylehint_uq_jp_photo_250220_1563413_r-600-800
Downloaded image 2 from https://api.fastretailing.com/ugc/v1/uq/jp/SR_IMAGES/ugc_stylehint_uq_jp_photo_250220_1563507_r-600-800
Downloaded image 7 from https://api.fastretailing.com/ugc/v1/uq/jp/SR_IMAGES/ugc_stylehint_uq_jp_photo_2

In [ ]:
!pip install torch torchvision
!pip install git+https://github.com/openai/CLIP.git
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 111.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 78.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

# Detect image product by clip

In [ ]:
import torch
import clip
from PIL import Image
import os

# 1. 加載更高分辨率的CLIP模型
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/16", device=device)  # 使用更高分辨率的模型

# 2. 定義更詳細的文本提示
categories = {
    "上衣": [
        "a white shirt", "a blue denim jacket", "a black t-shirt", "a red hoodie",
        "a striped sweater", "a leather jacket", "a floral blouse"
    ],
    "下身": [
        "black suit pants", "blue jeans", "dark blue jeans", "light blue jeans",
        "black shorts", "gray trousers", "beige chinos", "plaid trousers"
    ],
    "飾物": [
        "a fisherman hat", "a black belt", "a silver necklace", "a watch",
        "a scarf", "sunglasses", "a beanie", "a backpack"
    ],
    "季節性": ["spring summer", "autumn winter", "all season"],
    "場合": ["casual office wear", "formal event", "sports activity", "daily casual"],
    "相片感覺": ["vibrant and energetic", "calm and peaceful", "elegant and sophisticated"],
    "風格": ["innovative fashion", "classic style", "streetwear", "minimalist"],
    "性別": ["men's clothing", "women's clothing", "unisex"],
    "穿搭建議": [
        "layered outfit with detailed accessories", "simple and clean look",
        "bold and colorful combination", "monochrome style"
    ],
    "其他": ["none"]
}

# 3. 遍歷目錄中的所有圖像文件
image_dir = "/content/images"  # 圖像目錄路徑
image_files = [f for f in os.listdir(image_dir) if f.endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif'))]

if not image_files:
    print("No images found in the directory.")
    exit()

# 4. 對每個圖像進行處理
for image_file in image_files:
    image_path = os.path.join(image_dir, image_file)
    try:
        # 打開圖像
        image = Image.open(image_path)
        image_input = preprocess(image).unsqueeze(0).to(device)

        # 5. 逐類別計算最匹配的描述
        results = {}
        for category, descriptions in categories.items():
            text_inputs = clip.tokenize(descriptions).to(device)
            with torch.no_grad():
                text_features = model.encode_text(text_inputs)
                image_features = model.encode_image(image_input)
                logits_per_image = (image_features @ text_features.T).softmax(dim=-1)
                probs = logits_per_image.cpu().numpy()

            best_match_index = probs.argmax()
            best_match_description = descriptions[best_match_index]
            results[category] = best_match_description

        # 6. 後處理結果（根據需要修正描述）
        # if results["下身"] == "gray trousers":
        #     results["下身"] = "dark blue jeans"
        # 7. 輸出結構化結果
        print(f"\nResults for image: {image_file}")
        for category, description in results.items():
            print(f"{category}: {description}")

    except Exception as e:
        print(f"Error processing image {image_file}: {e}")

100%|███████████████████████████████████████| 335M/335M [00:25<00:00, 13.5MiB/s]



Results for image: image_12.jpg
上衣: a white shirt
下身: gray trousers
飾物: a watch
季節性: spring summer
場合: casual office wear
相片感覺: elegant and sophisticated
風格: classic style
性別: men's clothing
穿搭建議: simple and clean look
其他: none

Results for image: image_51.jpg
上衣: a white shirt
下身: gray trousers
飾物: a black belt
季節性: spring summer
場合: casual office wear
相片感覺: elegant and sophisticated
風格: streetwear
性別: men's clothing
穿搭建議: simple and clean look
其他: none

Results for image: image_62.jpg
上衣: a white shirt
下身: gray trousers
飾物: a watch
季節性: spring summer
場合: casual office wear
相片感覺: elegant and sophisticated
風格: classic style
性別: men's clothing
穿搭建議: monochrome style
其他: none

Results for image: image_99.jpg
上衣: a blue denim jacket
下身: gray trousers
飾物: a watch
季節性: spring summer
場合: casual office wear
相片感覺: elegant and sophisticated
風格: classic style
性別: men's clothing
穿搭建議: monochrome style
其他: none

Results for image: image_65.jpg
上衣: a white shirt
下身: gray trousers
飾物: a watch
季節性: 

# Train by Yolo

In [ ]:
from ultralytics import YOLO

# Load a model
model = YOLO("yolo11n-cls.pt")  # load a pretrained model (recommended for training)

# Train the model
results = model.train(data="fashion-mnist", epochs=100, imgsz=28)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


100%|██████████| 5.52M/5.52M [00:00<00:00, 105MB/s]

Ultralytics 8.3.77 🚀 Python-3.11.11 torch-2.5.1+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=classify, mode=train, model=yolo11n-cls.pt, data=fashion-mnist, epochs=100, time=None, patience=100, batch=16, imgsz=28, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, show_boxes=True, 


100%|██████████| 47.0M/47.0M [00:00<00:00, 63.2MB/s]
Unzipping /content/datasets/fashion-mnist.zip to /content/datasets/fashion-mnist...: 100%|██████████| 70023/70023 [00:07<00:00, 9698.69file/s]

Dataset download success ✅ (9.6s), saved to /content/datasets/fashion-mnist



...

  0%|          | 0/3750 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/ultralytics/nn/modules/block.py:917: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at ../aten/src/ATen/Context.cpp:208.)
  attn = (q.transpose(-2, -1) @ k) * self.scale
/usr/local/lib/python3.11/dist-packages/ultralytics/nn/modules/block.py:919: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this

                   all      0.884      0.999



      Epoch    GPU_mem       loss  Instances       Size


  0%|          | 0/3750 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/ultralytics/nn/modules/block.py:917: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at ../aten/src/ATen/Context.cpp:208.)
  attn = (q.transpose(-2, -1) @ k) * self.scale
/usr/local/lib/python3.11/dist-packages/ultralytics/nn/modules/block.py:919: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this

KeyboardInterrupt: 